# Step 4｜Depth 0 Feature Mart


## 目的

以`train_base.parquet`为样本轴，使用DuckDB SQL连接Depth 0特征表，
构建一行一个`case_id`的开发期Feature Mart。

## 数据范围

- Model Train：WEEK_NUM 0–61
- Tuning Validation：WEEK_NUM 62–71
- Calibration/Policy Validation：WEEK_NUM 72–81
- Final OOT：WEEK_NUM 82–91，当前不进入特征开发

## 本阶段使用的表

- train_base.parquet
- train_static_0_0.parquet
- train_static_0_1.parquet
- train_static_cb_0.parquet

## 核心约束

- Feature Mart必须一行一个case_id；
- 不读取官方test数据；
- 不使用Final OOT进行特征选择；
- 先审计表粒度，再执行JOIN；
- JOIN前后必须检查行数变化；
- SQL负责读取、连接和质量检查；
- Python负责配置控制、审计和结果保存。

## 本阶段暂不做

- 不处理Depth 1和Depth 2表；
- 不进行特征筛选；
- 不填补缺失值；
- 不进行类别编码；
- 不训练任何模型。

In [1]:
# Step 4.1.2：安装Step 4需要的依赖

%pip install duckdb pyyaml

In [2]:
# Step 4.1.3：配置路径并验证冻结切分

from pathlib import Path

import duckdb
import yaml


# ==========================================
# 1. 项目路径
# ==========================================

DATA_ROOT = Path(r"D:\Home_Credit_datasets")
PROJECT_ROOT = Path(r"D:\Risk_control project")

TRAIN_DIR = (
    DATA_ROOT
    / "parquet_files"
    / "train"
)

CONFIG_PATH = (
    PROJECT_ROOT
    / "configs"
    / "split_v1.yaml"
)

SQL_DIR = PROJECT_ROOT / "sql"

TEMP_DIR = (
    PROJECT_ROOT
    / "data"
    / "temp"
    / "duckdb"
)

PROCESSED_DIR = (
    PROJECT_ROOT
    / "data"
    / "processed"
)


for directory in [
    SQL_DIR,
    TEMP_DIR,
    PROCESSED_DIR
]:
    directory.mkdir(
        parents=True,
        exist_ok=True
    )


# ==========================================
# 2. Depth 0文件路径
# ==========================================

TRAIN_BASE_PATH = (
    TRAIN_DIR
    / "train_base.parquet"
)

TRAIN_STATIC_PATHS = sorted(
    TRAIN_DIR.glob(
        "train_static_0_*.parquet"
    )
)

TRAIN_STATIC_CB_PATH = (
    TRAIN_DIR
    / "train_static_cb_0.parquet"
)


required_files = [
    TRAIN_BASE_PATH,
    *TRAIN_STATIC_PATHS,
    TRAIN_STATIC_CB_PATH,
    CONFIG_PATH
]

missing_files = [
    str(file_path)
    for file_path in required_files
    if not file_path.is_file()
]

if missing_files:
    raise FileNotFoundError(
        "以下必需文件不存在：\n"
        + "\n".join(missing_files)
    )


if len(TRAIN_STATIC_PATHS) != 2:
    raise ValueError(
        "预期找到2个train_static分片，"
        f"实际找到{len(TRAIN_STATIC_PATHS)}个"
    )


# ==========================================
# 3. 读取冻结切分配置
# ==========================================

with CONFIG_PATH.open(
    "r",
    encoding="utf-8"
) as config_file:

    split_config = yaml.safe_load(
        config_file
    )


if split_config["status"] != "frozen":
    raise ValueError(
        "split_v1.yaml尚未处于frozen状态"
    )


expected_boundaries = {
    "model_train": (0, 61),
    "tuning_validation": (62, 71),
    "calibration_policy_validation": (72, 81),
    "final_oot": (82, 91)
}


actual_boundaries = {
    split_name: (
        split_config["splits"][split_name]["start_week"],
        split_config["splits"][split_name]["end_week"]
    )
    for split_name in expected_boundaries
}


if actual_boundaries != expected_boundaries:
    raise ValueError(
        "split_v1.yaml中的边界与已冻结方案不一致"
    )


DEV_START_WEEK = (
    actual_boundaries["model_train"][0]
)

DEV_END_WEEK = (
    actual_boundaries[
        "calibration_policy_validation"
    ][1]
)

OOT_START_WEEK = (
    actual_boundaries["final_oot"][0]
)

OOT_END_WEEK = (
    actual_boundaries["final_oot"][1]
)


# ==========================================
# 4. 建立DuckDB连接
# ==========================================

connection = duckdb.connect(
    database=":memory:"
)

connection.execute(
    "SET memory_limit = '20GB'"
)

temp_directory_sql = (
    TEMP_DIR.as_posix()
    .replace("'", "''")
)

connection.execute(
    f"SET temp_directory = "
    f"'{temp_directory_sql}'"
)


# ==========================================
# 5. 验收输出
# ==========================================

print("DuckDB版本：", duckdb.__version__)

print("\n切分配置：")
print("split_id：", split_config["split_id"])
print("状态：", split_config["status"])
print(
    "开发期周范围：",
    DEV_START_WEEK,
    "至",
    DEV_END_WEEK
)
print(
    "Final OOT周范围：",
    OOT_START_WEEK,
    "至",
    OOT_END_WEEK
)

print("\nDepth 0输入文件：")
print("-", TRAIN_BASE_PATH.name)

for file_path in TRAIN_STATIC_PATHS:
    print("-", file_path.name)

print("-", TRAIN_STATIC_CB_PATH.name)

print("\n输出目录：")
print("SQL目录：", SQL_DIR)
print("临时目录：", TEMP_DIR)
print("处理后数据目录：", PROCESSED_DIR)

print("\nStep 4.1环境与文件检查通过。")

DuckDB版本： 1.5.5

切分配置：
split_id： split_v1
状态： frozen
开发期周范围： 0 至 81
Final OOT周范围： 82 至 91

Depth 0输入文件：
- train_base.parquet
- train_static_0_0.parquet
- train_static_0_1.parquet
- train_static_cb_0.parquet

输出目录：
SQL目录： D:\Risk_control project\sql
临时目录： D:\Risk_control project\data\temp\duckdb
处理后数据目录： D:\Risk_control project\data\processed

Step 4.1环境与文件检查通过。


In [3]:
# Step 4.2.1：建立开发期DuckDB视图

def sql_path(path: Path) -> str:
    """
    将Windows路径转换成DuckDB可识别的SQL字符串。
    """
    path_text = (
        path.as_posix()
        .replace("'", "''")
    )

    return f"'{path_text}'"


base_path_sql = sql_path(
    TRAIN_BASE_PATH
)

static_paths_sql = ", ".join(
    sql_path(file_path)
    for file_path in TRAIN_STATIC_PATHS
)

static_cb_path_sql = sql_path(
    TRAIN_STATIC_CB_PATH
)


# ==========================================
# 1. 开发期Base视图
# ==========================================

connection.execute(
    f"""
    CREATE OR REPLACE VIEW base_dev AS

    SELECT
        case_id,
        date_decision,
        WEEK_NUM,
        MONTH,
        target

    FROM read_parquet(
        {base_path_sql}
    )

    WHERE WEEK_NUM BETWEEN
        {DEV_START_WEEK}
        AND
        {DEV_END_WEEK}
    """
)


# ==========================================
# 2. 两个Static分片的原始联合视图
# ==========================================

connection.execute(
    f"""
    CREATE OR REPLACE VIEW static_raw AS

    SELECT *

    FROM read_parquet(
        [{static_paths_sql}],
        union_by_name = true,
        filename = true
    )
    """
)


# 只保留开发期case_id

connection.execute(
    """
    CREATE OR REPLACE VIEW static_dev AS

    SELECT
        static_data.*

    FROM static_raw AS static_data

    INNER JOIN base_dev AS base
        ON static_data.case_id = base.case_id
    """
)


# ==========================================
# 3. Static CB开发期视图
# ==========================================

connection.execute(
    f"""
    CREATE OR REPLACE VIEW static_cb_raw AS

    SELECT *

    FROM read_parquet(
        {static_cb_path_sql},
        filename = true
    )
    """
)


connection.execute(
    """
    CREATE OR REPLACE VIEW static_cb_dev AS

    SELECT
        static_cb.*

    FROM static_cb_raw AS static_cb

    INNER JOIN base_dev AS base
        ON static_cb.case_id = base.case_id
    """
)


print("开发期视图创建成功：")

for view_name in [
    "base_dev",
    "static_dev",
    "static_cb_dev"
]:
    print("-", view_name)

开发期视图创建成功：
- base_dev
- static_dev
- static_cb_dev


In [4]:
# Step 4.2.2：确认base_dev没有包含Final OOT

dev_time_audit = connection.execute(
    """
    SELECT
        MIN(WEEK_NUM) AS minimum_week,
        MAX(WEEK_NUM) AS maximum_week,
        COUNT(DISTINCT WEEK_NUM) AS week_count,
        COUNT(*) AS sample_count,
        SUM(target) AS event_count,
        AVG(target) AS event_rate

    FROM base_dev
    """
).df()


display(dev_time_audit)


expected_dev_sample_count = sum(
    split_config["splits"][split_name]["sample_count"]
    for split_name in [
        "model_train",
        "tuning_validation",
        "calibration_policy_validation"
    ]
)

actual_dev_sample_count = int(
    dev_time_audit.loc[
        0,
        "sample_count"
    ]
)


if actual_dev_sample_count != expected_dev_sample_count:
    raise ValueError(
        "开发期样本数与split_v1.yaml不一致："
        f"预期{expected_dev_sample_count}，"
        f"实际{actual_dev_sample_count}"
    )


if int(dev_time_audit.loc[0, "maximum_week"]) >= OOT_START_WEEK:
    raise ValueError(
        "base_dev意外包含Final OOT数据"
    )


print(
    "开发期样本数验证通过：",
    actual_dev_sample_count
)

print(
    "Final OOT未进入base_dev：",
    True
)

,minimum_week,maximum_week,week_count,sample_count,event_count,event_rate
0,0,81,82,1401854,45394.0,0.032381


开发期样本数验证通过： 1401854
Final OOT未进入base_dev： True


In [5]:
# Step 4.2.3：审计Depth 0表的case_id粒度

import pandas as pd


audit_views = {
    "base_dev": "base_dev",
    "static_dev": "static_dev",
    "static_cb_dev": "static_cb_dev"
}


audit_results = []


for dataset_name, view_name in audit_views.items():

    audit_result = connection.execute(
        f"""
        WITH case_counts AS (
            SELECT
                case_id,
                COUNT(*) AS records_per_case

            FROM {view_name}

            GROUP BY case_id
        )

        SELECT
            '{dataset_name}' AS dataset,

            CAST(
                COALESCE(
                    SUM(records_per_case),
                    0
                )
                AS BIGINT
            ) AS row_count,

            CAST(
                COUNT(*) FILTER (
                    WHERE case_id IS NOT NULL
                )
                AS BIGINT
            ) AS unique_case_id_count,

            CAST(
                COALESCE(
                    SUM(
                        CASE
                            WHEN case_id IS NULL
                            THEN records_per_case
                            ELSE 0
                        END
                    ),
                    0
                )
                AS BIGINT
            ) AS null_case_id_rows,

            CAST(
                COALESCE(
                    SUM(
                        CASE
                            WHEN case_id IS NOT NULL
                                 AND records_per_case > 1
                            THEN 1
                            ELSE 0
                        END
                    ),
                    0
                )
                AS BIGINT
            ) AS duplicate_case_id_count,

            CAST(
                COALESCE(
                    SUM(
                        CASE
                            WHEN case_id IS NOT NULL
                                 AND records_per_case > 1
                            THEN records_per_case - 1
                            ELSE 0
                        END
                    ),
                    0
                )
                AS BIGINT
            ) AS duplicate_extra_rows

        FROM case_counts
        """
    ).df()

    audit_results.append(
        audit_result
    )


case_id_audit = pd.concat(
    audit_results,
    ignore_index=True
)


display(case_id_audit)

,dataset,row_count,unique_case_id_count,null_case_id_rows,duplicate_case_id_count,duplicate_extra_rows
0,base_dev,1401854,1401854,0,0,0
1,static_dev,1401854,1401854,0,0,0
2,static_cb_dev,1375925,1375925,0,0,0


In [6]:
# Step 4.2.3验收

grain_problems = case_id_audit[
    (case_id_audit["null_case_id_rows"] > 0)
    |
    (case_id_audit["duplicate_case_id_count"] > 0)
    |
    (case_id_audit["duplicate_extra_rows"] > 0)
]


if not grain_problems.empty:
    display(grain_problems)

    raise ValueError(
        "至少一张Depth 0表不满足"
        "一行一个case_id"
    )


print(
    "Depth 0主键粒度检查通过："
    "三张开发期视图均为一行一个case_id。"
)

Depth 0主键粒度检查通过：三张开发期视图均为一行一个case_id。


In [7]:
# Step 4.2.4：查看两个Static分片各自贡献的行数

static_shard_audit = connection.execute(
    """
    SELECT
        filename AS source_file,
        COUNT(*) AS row_count,
        COUNT(DISTINCT case_id)
            AS unique_case_id_count

    FROM static_dev

    GROUP BY filename

    ORDER BY filename
    """
).df()


display(static_shard_audit)

,source_file,row_count,unique_case_id_count
0,D:/Home_Credit_datasets/parquet_files/train/tr...,1003757,1003757
1,D:/Home_Credit_datasets/parquet_files/train/tr...,398097,398097


In [8]:
# Step 4.3.1：Depth 0总体覆盖率审计

coverage_audit = connection.execute(
    """
    SELECT
        'static_dev' AS feature_source,

        COUNT(*) AS base_case_count,

        COUNT(static_data.case_id)
            AS matched_case_count,

        COUNT(*) - COUNT(static_data.case_id)
            AS unmatched_case_count,

        ROUND(
            100.0
            * COUNT(static_data.case_id)
            / COUNT(*),
            4
        ) AS coverage_rate_pct

    FROM base_dev AS base

    LEFT JOIN static_dev AS static_data
        ON base.case_id = static_data.case_id


    UNION ALL


    SELECT
        'static_cb_dev' AS feature_source,

        COUNT(*) AS base_case_count,

        COUNT(static_cb.case_id)
            AS matched_case_count,

        COUNT(*) - COUNT(static_cb.case_id)
            AS unmatched_case_count,

        ROUND(
            100.0
            * COUNT(static_cb.case_id)
            / COUNT(*),
            4
        ) AS coverage_rate_pct

    FROM base_dev AS base

    LEFT JOIN static_cb_dev AS static_cb
        ON base.case_id = static_cb.case_id
    """
).df()


display(coverage_audit)

,feature_source,base_case_count,matched_case_count,unmatched_case_count,coverage_rate_pct
0,static_dev,1401854,1401854,0,100.0000
1,static_cb_dev,1401854,1375925,25929,98.1504


In [9]:
# ==========================================
# 关键检查
# ==========================================

static_result = (
    coverage_audit
    .loc[
        coverage_audit["feature_source"]
        == "static_dev"
    ]
    .iloc[0]
)

static_cb_result = (
    coverage_audit
    .loc[
        coverage_audit["feature_source"]
        == "static_cb_dev"
    ]
    .iloc[0]
)


critical_checks = {
    "base样本数与split配置一致": (
        coverage_audit[
            "base_case_count"
        ].eq(
            expected_dev_sample_count
        ).all()
    ),

    "覆盖率加总关系正确": (
        (
            coverage_audit[
                "matched_case_count"
            ]
            +
            coverage_audit[
                "unmatched_case_count"
            ]
        )
        .eq(
            coverage_audit[
                "base_case_count"
            ]
        )
        .all()
    ),

    "Static完整覆盖开发期Base": (
        int(
            static_result[
                "matched_case_count"
            ]
        )
        == expected_dev_sample_count
        and
        int(
            static_result[
                "unmatched_case_count"
            ]
        )
        == 0
    ),

    "覆盖率位于0%至100%": (
        coverage_audit[
            "coverage_rate_pct"
        ]
        .between(
            0,
            100
        )
        .all()
    )
}


failed_checks = [
    check_name
    for check_name, passed
    in critical_checks.items()
    if not passed
]


if failed_checks:
    raise ValueError(
        "Step 4.3关键检查失败：\n- "
        + "\n- ".join(failed_checks)
    )


print(
    "Step 4.3关键检查："
    f"{len(critical_checks)}/"
    f"{len(critical_checks)}通过"
)

print(
    "Static CB未匹配案件数：",
    int(
        static_cb_result[
            "unmatched_case_count"
        ]
    )
)

print(
    "Static CB总体覆盖率：",
    f"{float(static_cb_result['coverage_rate_pct']):.4f}%"
)

Step 4.3关键检查：4/4通过
Static CB未匹配案件数： 25929
Static CB总体覆盖率： 98.1504%


In [10]:
# Step 4.3.2：Static CB覆盖率时间变化自动检查

train_start_week = (
    split_config["splits"]
    ["model_train"]
    ["start_week"]
)

train_end_week = (
    split_config["splits"]
    ["model_train"]
    ["end_week"]
)

validation_start_week = (
    split_config["splits"]
    ["tuning_validation"]
    ["start_week"]
)

validation_end_week = (
    split_config["splits"]
    ["tuning_validation"]
    ["end_week"]
)

calibration_start_week = (
    split_config["splits"]
    ["calibration_policy_validation"]
    ["start_week"]
)

calibration_end_week = (
    split_config["splits"]
    ["calibration_policy_validation"]
    ["end_week"]
)


static_cb_split_coverage = connection.execute(
    f"""
    WITH split_map AS (
        SELECT
            *

        FROM (
            VALUES
                (
                    'Model Train',
                    1,
                    {train_start_week},
                    {train_end_week}
                ),
                (
                    'Tuning Validation',
                    2,
                    {validation_start_week},
                    {validation_end_week}
                ),
                (
                    'Calibration/Policy Validation',
                    3,
                    {calibration_start_week},
                    {calibration_end_week}
                )
        ) AS splits(
            split_name,
            split_order,
            start_week,
            end_week
        )
    ),

    base_with_split AS (
        SELECT
            base.case_id,
            base.WEEK_NUM,
            splits.split_name,
            splits.split_order

        FROM base_dev AS base

        INNER JOIN split_map AS splits
            ON base.WEEK_NUM
            BETWEEN splits.start_week
                AND splits.end_week
    )

    SELECT
        base.split_name,
        base.split_order,

        COUNT(*) AS base_case_count,

        COUNT(static_cb.case_id)
            AS matched_case_count,

        COUNT(*) - COUNT(static_cb.case_id)
            AS unmatched_case_count,

        ROUND(
            100.0
            * COUNT(static_cb.case_id)
            / COUNT(*),
            4
        ) AS coverage_rate_pct

    FROM base_with_split AS base

    LEFT JOIN static_cb_dev AS static_cb
        ON base.case_id = static_cb.case_id

    GROUP BY
        base.split_name,
        base.split_order

    ORDER BY
        base.split_order
    """
).df()


# ==========================================
# 异常触发规则
# ==========================================

COVERAGE_ALERT_THRESHOLD_PP = 1.0

maximum_coverage = (
    static_cb_split_coverage[
        "coverage_rate_pct"
    ].max()
)

minimum_coverage = (
    static_cb_split_coverage[
        "coverage_rate_pct"
    ].min()
)

coverage_range_pp = (
    maximum_coverage
    - minimum_coverage
)


print(
    "Static CB跨阶段覆盖率极差：",
    f"{coverage_range_pp:.4f}个百分点"
)

print(
    "预警阈值：",
    f"{COVERAGE_ALERT_THRESHOLD_PP:.4f}个百分点"
)


if coverage_range_pp > COVERAGE_ALERT_THRESHOLD_PP:

    print(
        "\n触发覆盖率时间变化预警，"
        "显示详细结果："
    )

    display(
        static_cb_split_coverage.drop(
            columns="split_order"
        )
    )

else:

    print(
        "未触发预警，"
        "不展开分阶段明细。"
    )

Static CB跨阶段覆盖率极差： 2.0535个百分点
预警阈值： 1.0000个百分点

触发覆盖率时间变化预警，显示详细结果：


,split_name,base_case_count,matched_case_count,unmatched_case_count,coverage_rate_pct
0,Model Train,1250581,1224704,25877,97.9308
1,Tuning Validation,63875,63865,10,99.9843
2,Calibration/Policy Validation,87398,87356,42,99.9519


## Step 4.3结论｜Static CB覆盖率变化

Static表完整覆盖开发期Base。

Static CB覆盖率存在时间变化：

- Model Train：97.9308%
- Tuning Validation：99.9843%
- Calibration/Policy Validation：99.9519%

跨阶段覆盖率极差为2.0535个百分点，超过1个百分点预警线。

该现象说明Static CB记录的可用性随时间变化，但目前没有证据证明数据错误。
因此：

- 不删除缺少Static CB记录的案件；
- 不修改已经冻结的时间切分；
- 使用LEFT JOIN保留全部Base案件；
- 缺少整条Static CB记录时，其对应特征保留为空；
- 增加has_static_cb_record来源标记；
- 该标记暂作为审计字段，初始模型不直接将其作为预测变量；
- 后续比较Static CB特征加入前后的时间稳定性。

In [11]:
# Step 4.4.1：自动检查字段重名

base_schema = connection.execute(
    "DESCRIBE base_dev"
).df()

static_schema = connection.execute(
    "DESCRIBE static_dev"
).df()

static_cb_schema = connection.execute(
    "DESCRIBE static_cb_dev"
).df()


base_columns = (
    base_schema["column_name"]
    .tolist()
)

static_columns = (
    static_schema["column_name"]
    .tolist()
)

static_cb_columns = (
    static_cb_schema["column_name"]
    .tolist()
)


# 连接时不会重复选择这些技术字段

excluded_static_columns = {
    "case_id",
    "filename"
}

excluded_static_cb_columns = {
    "case_id",
    "filename"
}


static_feature_columns = [
    column_name
    for column_name in static_columns
    if column_name
    not in excluded_static_columns
]

static_cb_feature_columns = [
    column_name
    for column_name in static_cb_columns
    if column_name
    not in excluded_static_cb_columns
]


# 检查真正进入宽表的字段是否重名

static_base_collisions = sorted(
    set(static_feature_columns)
    & set(base_columns)
)

static_cb_base_collisions = sorted(
    set(static_cb_feature_columns)
    & set(base_columns)
)

feature_table_collisions = sorted(
    set(static_feature_columns)
    & set(static_cb_feature_columns)
)

reserved_name_collisions = sorted(
    {
        "has_static_cb_record"
    }
    & (
        set(base_columns)
        | set(static_feature_columns)
        | set(static_cb_feature_columns)
    )
)


collision_summary = {
    "Static与Base重名字段":
        static_base_collisions,

    "Static CB与Base重名字段":
        static_cb_base_collisions,

    "Static与Static CB重名字段":
        feature_table_collisions,

    "审计标记字段重名":
        reserved_name_collisions
}


detected_collisions = {
    collision_type: columns
    for collision_type, columns
    in collision_summary.items()
    if columns
}


if detected_collisions:

    print("检测到字段重名：")

    for collision_type, columns in (
        detected_collisions.items()
    ):
        print(
            f"- {collision_type}：",
            columns
        )

    raise ValueError(
        "存在字段重名，"
        "暂不构建Depth 0宽表。"
    )


expected_feature_mart_column_count = (
    len(base_columns)
    + len(static_feature_columns)
    + len(static_cb_feature_columns)
    + 1
)


print("字段重名检查通过。")
print("Base字段数：", len(base_columns))
print(
    "Static特征字段数：",
    len(static_feature_columns)
)
print(
    "Static CB特征字段数：",
    len(static_cb_feature_columns)
)
print(
    "预计宽表字段总数：",
    expected_feature_mart_column_count
)

字段重名检查通过。
Base字段数： 5
Static特征字段数： 167
Static CB特征字段数： 52
预计宽表字段总数： 225


In [12]:
# Step 4.4.2：构建正式Depth 0开发期宽表

connection.execute(
    """
    CREATE OR REPLACE VIEW
        depth0_feature_mart_dev AS

    SELECT
        base.*,

        CASE
            WHEN static_cb.case_id IS NOT NULL
            THEN 1
            ELSE 0
        END AS has_static_cb_record,

        static_data.* EXCLUDE (
            case_id,
            filename
        ),

        static_cb.* EXCLUDE (
            case_id,
            filename
        )

    FROM base_dev AS base

    LEFT JOIN static_dev AS static_data
        ON base.case_id = static_data.case_id

    LEFT JOIN static_cb_dev AS static_cb
        ON base.case_id = static_cb.case_id
    """
)


print(
    "Depth 0正式宽表视图创建成功："
    "depth0_feature_mart_dev"
)

Depth 0正式宽表视图创建成功：depth0_feature_mart_dev


In [13]:
# Step 4.4.3：正式宽表连接后完整性检查

feature_mart_audit = connection.execute(
    """
    SELECT
        COUNT(*) AS row_count,

        COUNT(DISTINCT case_id)
            AS unique_case_id_count,

        SUM(
            CASE
                WHEN case_id IS NULL
                THEN 1
                ELSE 0
            END
        ) AS null_case_id_count,

        COUNT(*) - COUNT(DISTINCT case_id)
            AS duplicate_extra_rows,

        MIN(WEEK_NUM) AS minimum_week,

        MAX(WEEK_NUM) AS maximum_week,

        SUM(
            CASE
                WHEN has_static_cb_record = 0
                THEN 1
                ELSE 0
            END
        ) AS missing_static_cb_record_count

    FROM depth0_feature_mart_dev
    """
).df()


actual_feature_mart_column_count = len(
    connection.execute(
        """
        DESCRIBE depth0_feature_mart_dev
        """
    ).df()
)


display(feature_mart_audit)

print(
    "实际宽表字段总数：",
    actual_feature_mart_column_count
)

,row_count,unique_case_id_count,null_case_id_count,duplicate_extra_rows,minimum_week,maximum_week,missing_static_cb_record_count
0,1401854,1401854,0.0,0,0,81,25929.0


实际宽表字段总数： 225


In [14]:
# Step 4.4关键验收

audit_row = feature_mart_audit.iloc[0]

actual_row_count = int(
    audit_row["row_count"]
)

actual_unique_case_count = int(
    audit_row["unique_case_id_count"]
)

actual_null_case_count = int(
    audit_row["null_case_id_count"]
)

actual_duplicate_extra_rows = int(
    audit_row["duplicate_extra_rows"]
)

actual_minimum_week = int(
    audit_row["minimum_week"]
)

actual_maximum_week = int(
    audit_row["maximum_week"]
)

actual_missing_static_cb_count = int(
    audit_row[
        "missing_static_cb_record_count"
    ]
)

expected_missing_static_cb_count = int(
    static_cb_result[
        "unmatched_case_count"
    ]
)


critical_checks = {
    "宽表行数保持不变": (
        actual_row_count
        == expected_dev_sample_count
    ),

    "一行一个case_id": (
        actual_unique_case_count
        == expected_dev_sample_count
    ),

    "case_id不存在缺失": (
        actual_null_case_count == 0
    ),

    "连接没有产生额外行": (
        actual_duplicate_extra_rows == 0
    ),

    "宽表不包含Final OOT": (
        actual_minimum_week
        == DEV_START_WEEK
        and
        actual_maximum_week
        == DEV_END_WEEK
    ),

    "Static CB缺失记录数一致": (
        actual_missing_static_cb_count
        == expected_missing_static_cb_count
    ),

    "宽表字段数量符合预期": (
        actual_feature_mart_column_count
        == expected_feature_mart_column_count
    )
}


failed_checks = [
    check_name
    for check_name, passed
    in critical_checks.items()
    if not passed
]


if failed_checks:
    raise ValueError(
        "Step 4.4关键检查失败：\n- "
        + "\n- ".join(failed_checks)
    )


print(
    "Step 4.4关键检查："
    f"{len(critical_checks)}/"
    f"{len(critical_checks)}通过"
)

print(
    "Depth 0宽表保持"
    f"{actual_row_count:,}行，"
    "一行一个case_id。"
)

print(
    "Static CB整条记录缺失案件数：",
    f"{actual_missing_static_cb_count:,}"
)

Step 4.4关键检查：7/7通过
Depth 0宽表保持1,401,854行，一行一个case_id。
Static CB整条记录缺失案件数： 25,929


In [15]:
# Step 4.5.1：导出Depth 0开发期Feature Mart

FEATURE_MART_PATH = (
    PROCESSED_DIR
    / "depth0_feature_mart_dev_v1.parquet"
)

feature_mart_path_sql = sql_path(
    FEATURE_MART_PATH
)


if FEATURE_MART_PATH.exists():

    print(
        "Feature Mart文件已经存在，"
        "跳过重复导出。"
    )

else:

    connection.execute(
        f"""
        COPY (
            SELECT *

            FROM depth0_feature_mart_dev
        )

        TO {feature_mart_path_sql}

        (
            FORMAT PARQUET,
            COMPRESSION ZSTD,
            ROW_GROUP_SIZE 100000
        )
        """
    )

    print("Feature Mart导出成功。")


print("输出文件：", FEATURE_MART_PATH)
print(
    "文件是否存在：",
    FEATURE_MART_PATH.exists()
)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Feature Mart导出成功。
输出文件： D:\Risk_control project\data\processed\depth0_feature_mart_dev_v1.parquet
文件是否存在： True


In [16]:
# Step 4.5.2：验证导出的Parquet文件

export_audit = connection.execute(
    f"""
    SELECT
        COUNT(*) AS row_count,

        COUNT(DISTINCT case_id)
            AS unique_case_id_count,

        MIN(WEEK_NUM) AS minimum_week,

        MAX(WEEK_NUM) AS maximum_week,

        SUM(
            CASE
                WHEN has_static_cb_record = 0
                THEN 1
                ELSE 0
            END
        ) AS missing_static_cb_record_count

    FROM read_parquet(
        {feature_mart_path_sql}
    )
    """
).df()


export_column_count = len(
    connection.execute(
        f"""
        DESCRIBE
        SELECT *

        FROM read_parquet(
            {feature_mart_path_sql}
        )
        """
    ).df()
)


export_row = export_audit.iloc[0]


export_checks = {
    "导出行数正确": (
        int(export_row["row_count"])
        == expected_dev_sample_count
    ),

    "case_id数量正确": (
        int(export_row["unique_case_id_count"])
        == expected_dev_sample_count
    ),

    "没有包含Final OOT": (
        int(export_row["minimum_week"])
        == DEV_START_WEEK
        and
        int(export_row["maximum_week"])
        == DEV_END_WEEK
    ),

    "Static CB记录缺失数正确": (
        int(
            export_row[
                "missing_static_cb_record_count"
            ]
        )
        == actual_missing_static_cb_count
    ),

    "导出字段数正确": (
        export_column_count
        == expected_feature_mart_column_count
    )
}


failed_export_checks = [
    check_name
    for check_name, passed
    in export_checks.items()
    if not passed
]


if failed_export_checks:
    raise ValueError(
        "Feature Mart导出检查失败：\n- "
        + "\n- ".join(
            failed_export_checks
        )
    )


file_size_gb = (
    FEATURE_MART_PATH.stat().st_size
    / 1024**3
)


print(
    "Feature Mart导出检查："
    f"{len(export_checks)}/"
    f"{len(export_checks)}通过"
)

print(
    "文件大小：",
    f"{file_size_gb:.3f} GB"
)

print(
    "数据形状：",
    (
        expected_dev_sample_count,
        export_column_count
    )
)

Feature Mart导出检查：5/5通过
文件大小： 0.142 GB
数据形状： (1401854, 225)


In [17]:
# Step 4.5.3：保存Depth 0宽表SQL

DEPTH0_SQL_PATH = (
    SQL_DIR
    / "01_depth0_feature_mart_v1.sql"
)


depth0_sql_text = """
-- Depth 0 development Feature Mart
--
-- Grain:
--   One row per case_id.
--
-- Required input views:
--   base_dev
--   static_dev
--   static_cb_dev
--
-- Scope:
--   WEEK_NUM 0-81 only.
--   Final OOT weeks 82-91 are excluded.
--
-- Join policy:
--   base_dev defines the sample population.
--   LEFT JOIN preserves cases without Static CB records.

CREATE OR REPLACE VIEW depth0_feature_mart_dev AS

SELECT
    base.*,

    CASE
        WHEN static_cb.case_id IS NOT NULL
        THEN 1
        ELSE 0
    END AS has_static_cb_record,

    static_data.* EXCLUDE (
        case_id,
        filename
    ),

    static_cb.* EXCLUDE (
        case_id,
        filename
    )

FROM base_dev AS base

LEFT JOIN static_dev AS static_data
    ON base.case_id = static_data.case_id

LEFT JOIN static_cb_dev AS static_cb
    ON base.case_id = static_cb.case_id;
""".strip() + "\n"


if DEPTH0_SQL_PATH.exists():

    existing_sql_text = (
        DEPTH0_SQL_PATH.read_text(
            encoding="utf-8"
        )
    )

    if existing_sql_text != depth0_sql_text:
        raise FileExistsError(
            "SQL文件已存在且内容不同。"
            "如需改变逻辑，应创建v2，"
            "不要直接覆盖v1。"
        )

    print(
        "SQL文件已经存在，"
        "且内容一致。"
    )

else:

    DEPTH0_SQL_PATH.write_text(
        depth0_sql_text,
        encoding="utf-8"
    )

    print("SQL文件创建成功。")


print("SQL文件：", DEPTH0_SQL_PATH)

SQL文件创建成功。
SQL文件： D:\Risk_control project\sql\01_depth0_feature_mart_v1.sql


In [18]:
# Step 4.5.4：保存Feature Mart版本清单

from datetime import datetime, timezone


METADATA_DIR = (
    PROJECT_ROOT
    / "metadata"
)

METADATA_DIR.mkdir(
    parents=True,
    exist_ok=True
)

FEATURE_MART_MANIFEST_PATH = (
    METADATA_DIR
    / "depth0_feature_mart_v1.yaml"
)


manifest_core = {
    "feature_mart_id":
        "depth0_feature_mart_dev_v1",

    "split_id":
        split_config["split_id"],

    "grain":
        "one_row_per_case_id",

    "week_range": {
        "start_week":
            DEV_START_WEEK,

        "end_week":
            DEV_END_WEEK
    },

    "final_oot_included":
        False,

    "row_count":
        expected_dev_sample_count,

    "column_count":
        export_column_count,

    "static_cb_missing_record_count":
        actual_missing_static_cb_count,

    "has_static_cb_record_role":
        "audit_only_initially",

    "source_views": [
        "base_dev",
        "static_dev",
        "static_cb_dev"
    ],

    "data_file":
        FEATURE_MART_PATH
        .relative_to(PROJECT_ROOT)
        .as_posix(),

    "sql_file":
        DEPTH0_SQL_PATH
        .relative_to(PROJECT_ROOT)
        .as_posix()
}


if FEATURE_MART_MANIFEST_PATH.exists():

    with FEATURE_MART_MANIFEST_PATH.open(
        "r",
        encoding="utf-8"
    ) as manifest_file:

        existing_manifest = yaml.safe_load(
            manifest_file
        )

    existing_core = {
        key: existing_manifest.get(key)
        for key in manifest_core
    }

    if existing_core != manifest_core:
        raise FileExistsError(
            "现有Feature Mart清单"
            "与当前结果不同。"
            "需要创建新版本，不能覆盖v1。"
        )

    print(
        "Feature Mart清单已经存在，"
        "且内容一致。"
    )

else:

    feature_mart_manifest = {
        **manifest_core,

        "created_at_utc": (
            datetime.now(timezone.utc)
            .isoformat()
        )
    }

    with FEATURE_MART_MANIFEST_PATH.open(
        "w",
        encoding="utf-8"
    ) as manifest_file:

        yaml.safe_dump(
            feature_mart_manifest,
            manifest_file,
            allow_unicode=True,
            sort_keys=False
        )

    print("Feature Mart清单创建成功。")


print(
    "Feature Mart清单：",
    FEATURE_MART_MANIFEST_PATH
)

Feature Mart清单创建成功。
Feature Mart清单： D:\Risk_control project\metadata\depth0_feature_mart_v1.yaml
